# Tool Calling Fine-Tuning: DoRA (Weight-Decomposed LoRA, 16-bit) on Tesla T4 GPU
Bu notebook, **Qwen2.5-0.5B** modeli üzerinde **DoRA (Weight-Decomposed Low-Rank Adaptation, 16-bit)** yöntemiyle Tool / Function Calling yeteneğini eğitmek ve değerlendirmek için optimize edilmiştir.

- **Metot:** DoRA (`use_dora=True`, r=16, alpha=32, target_modules=[q_proj, k_proj, v_proj, o_proj])
- **Taban Model:** Qwen/Qwen2.5-0.5B (16-bit float16)
- **Hedef Donanım:** Google Colab Tesla T4 (~15GB VRAM, float16)
- **Sekans Uzunluğu:** max_seq_len = 2048 (Sistem şeması ve format yönergeleri korunur)
- **DoRA Mekanizması:** Ağırlık matrisini genlik (*Magnitude*) ve yönelim (*Direction*) bileşenlerine ayırarak standart LoRA'daki format yönelim kaybını engeller ve Full FT dinamiklerine en yakın davranışı sergiler.

---

### 1. GPU ve Donanım Kontrolü

In [ ]:
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise SystemError(
        "HATA: GPU runtime bulunamadi!\n"
        "Lutfen Google Colab'de 'T4 GPU' veya Kaggle'da 'GPU T4 x2' secin!"
    )

device_count = torch.cuda.device_count()
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Tespit Edilen GPU Sayisi: {device_count}")
for i in range(device_count):
    vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f"  [GPU {i}] {torch.cuda.get_device_name(i)} - {vram:.1f} GB VRAM")
print(f"bf16 destegi: {torch.cuda.is_bf16_supported()} (T4 icin False olmasi normaldir; float16 kullanilacaktir)")
if device_count > 1:
    print("\n[BILGI] Kaggle 2x T4 ortamindasiniz. Qwen2.5-0.5B modeli ~4-5 GB VRAM gerektirdigi icin")
    print("       egitim CUDA_VISIBLE_DEVICES=0 ile 1. GPU uzerinde kararlı ve en hizli sekilde calistirilacaktir.")


### 2. Projeyi Klonla ve Çalışma Dizinine Geç

In [ ]:
import os

REPO_URL = "https://github.com/fatihkadim/tool-calling-ft.git"

# Calisma ortami tespiti: Google Colab (/content) veya Kaggle (/kaggle/working)
if os.path.exists("/content"):
    PROJECT_DIR = "/content/tool-calling-ft"
elif os.path.exists("/kaggle/working"):
    PROJECT_DIR = "/kaggle/working/tool-calling-ft"
else:
    PROJECT_DIR = os.path.abspath(".")

if os.path.exists("/content") or os.path.exists("/kaggle/working"):
    if not os.path.exists(PROJECT_DIR):
        print(f"Repo klonlaniyor: {PROJECT_DIR}")
        !git clone {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}
    !git pull origin main
else:
    print("Yerel/Ozel calisma ortami:", os.getcwd())

!pwd


### 3. Bağımlılıkların Kurulumu ve Ortam Hazırlığı

In [ ]:
import os
import sys

# src dizinini Python import arama yoluna ekle
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

!pip install -q --upgrade pip
!pip uninstall -y torchao 2>/dev/null || true
!pip install -q "transformers>=4.46" "peft>=0.13" "bitsandbytes>=0.44" "datasets>=3.0" "trl>=0.11" "accelerate>=1.0" pyyaml tqdm pandas matplotlib

# uv_build backend'ini kur ve projeyi editable modda bağlamayı dene
!pip install -q "uv_build>=0.11.7,<0.12.0" 2>/dev/null || true
!pip install -q --no-build-isolation -e . 2>/dev/null || echo "Editable install atlandı, PYTHONPATH=src kullanılacak."

### 4. Veri Setini Kontrol Et ve Hazırla

In [ ]:
import os

train_path = "data/processed/train.jsonl"
eval_path = "data/processed/eval_subset.jsonl"

if not os.path.exists(train_path) or not os.path.exists(eval_path):
    print("İşlenmiş veri seti bulunamadı. HuggingFace'ten indirilip ChatML formatında hazırlanıyor...")
    !PYTHONPATH=src python -m tool_calling_ft.data.prepare_dataset
else:
    print(f"Veri seti hazır: {train_path} ve {eval_path} mevcut!")

### 5. T4 İçin DoRA Yapılandırması Oluştur

> **DoRA Nedir?** DoRA (Weight-Decomposed Low-Rank Adaptation), ağırlık güncellemesini *büyüklük (magnitude)* ve *yön (direction)* bileşenlerine ayırarak standart LoRA'ya kıyasla Full Fine-Tuning davranışına çok daha yakın, yönelim kararlılığı yüksek sonuçlar üretir.

In [ ]:
import os

import yaml

os.makedirs("configs", exist_ok=True)

config = {
    "method": "dora",
    "base_model": "Qwen/Qwen2.5-0.5B",
    "dataset": "NousResearch/hermes-function-calling-v1",
    "output_dir": "checkpoints/dora",
    "lora": {
        "r": 16,
        "alpha": 32,
        "dropout": 0.05,
        "use_dora": True,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    },
    "training": {
        "epochs": 3,
        "batch_size": 2,
        "grad_accum_steps": 4,
        "learning_rate": 2e-4,
        "max_seq_len": 2048,
        "warmup_ratio": 0.05,
        "save_steps": 200,
        "seed": 42,
    },
}

config_path = "configs/dora_t4.yaml"
with open(config_path, "w", encoding="utf-8") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"T4 DoRA konfigürasyonu kaydedildi: {config_path}")
print(yaml.dump(config, default_flow_style=False))

### 6. DoRA Eğitimini Başlat

In [ ]:
# T4 GPU üzerinde 16-bit DoRA eğitimi:
# - use_dora=True ile genlik ve yön dekompozisyonu uygulanır
# - expandable_segments:True bellek parçalanmasını önler
# - PYTHONPATH=src modül erişimini garanti eder
!PYTHONPATH=src PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True CUDA_VISIBLE_DEVICES=0 python -m tool_calling_ft.training.train --config configs/dora_t4.yaml

### 7. Canlı Demo (Inference Testi - Pozitif & Negatif Örnekler)

In [ ]:
import gc

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model_name = "Qwen/Qwen2.5-0.5B"
adapter_path = "checkpoints/dora"

print("Model ve DoRA adaptörü yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

system_prompt = """You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags.
<tools>
[{"type": "function", "function": {"name": "get_current_weather", "description": "Get current weather for a city", "parameters": {"type": "object", "properties": {"location": {"type": "string"}, "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}}, "required": ["location"]}}}]
</tools>
For each function call return a json object with function name and arguments within <tool_call> </tool_call> tags."""

test_cases = [
    ("Pozitif Örnek (Tool Çağrısı Beklenir):", "What is the weather in Tokyo in celsius?"),
    ("Negatif Örnek (Genel Sohbet / Tool Çağrılmamalı):", "What is the capital of France and what is it famous for?"),
]

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
stop_ids = [tokenizer.eos_token_id]
if isinstance(im_end_id, int) and im_end_id != tokenizer.eos_token_id:
    stop_ids.append(im_end_id)

print("=" * 60)
for label, query in test_cases:
    prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=stop_ids,
        )

    response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    print(f"\n{label}")
    print(f"Soru: {query}")
    print(f"Model Yanıtı:\n{response.strip()}")
    print("-" * 60)
print("=" * 60)

# Evaluation adımı için VRAM'i tamamen serbest bırak (OOM riskini önler)
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Demo tamamlandı, bellek (VRAM) temizlendi!")

### 8. DoRA Değerlendirme (Evaluation) & Metrik Tablosu

In [ ]:
import json
import os

import pandas as pd

# DoRA Modelini Değerlendir:
!PYTHONPATH=src CUDA_VISIBLE_DEVICES=0 python -m tool_calling_ft.eval.harness \
    --method dora \
    --adapter checkpoints/dora \
    --dataset data/processed/eval_subset.jsonl

# Sonuç JSON raporunu yükle ve görselleştir
report_path = "reports/dora_metrics.json"
if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        report = json.load(f)

    print("\n" + "=" * 55)
    print(" KALİTE METRİKLERİ (QUALITY METRICS)")
    print("=" * 55)
    df_quality = pd.DataFrame(list(report.get("quality_metrics", {}).items()), columns=["Metrik", "Değer"])
    display(df_quality)

    print("\n" + "=" * 55)
    print(" PERFORMANS & DONANIM METRİKLERİ")
    print("=" * 55)
    df_perf = pd.DataFrame(list(report.get("performance_metrics", {}).items()), columns=["Metrik", "Değer"])
    display(df_perf)
else:
    print("Uyarı: reports/dora_metrics.json bulunamadı.")

### 9. Sonuçları Yedekle (Google Drive veya ZIP İndirme)

In [ ]:
import os

# 1. Secenek: Google Colab Drive Yedekleme
if os.path.exists("/content"):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_dest = "/content/drive/MyDrive/tool_calling_dora_results"
        os.makedirs(drive_dest, exist_ok=True)
        !cp -r checkpoints/dora {drive_dest}/ 2>/dev/null || echo 'checkpoints klasoru kopyalanamadi'
        !cp -r reports {drive_dest}/ 2>/dev/null || echo 'reports klasoru kopyalanamadi'
        print(f"DoRA sonuclari Drive'a kaydedildi -> {drive_dest}")
    except Exception as e:
        print("Google Drive baglantisi atlandi:", e)

# 2. Secenek: ZIP arsivi (Hem Colab hem Kaggle ile %100 uyumlu)
!zip -q -r dora_results.zip checkpoints/dora reports 2>/dev/null || true
if os.path.exists("dora_results.zip"):
    if os.path.exists("/kaggle/working") and os.getcwd() != "/kaggle/working":
        !cp dora_results.zip /kaggle/working/
        print("dora_results.zip Kaggle /kaggle/working dizinine kopyalandi!")
        print("Notebook tamamlandiginda sag paneldeki 'Output' sekmesinden dogrudan indirebilirsiniz.")
    else:
        print("\ndora_results.zip arsivi olusturuldu! Dosyalar panelinden indirebilirsiniz.")
